In [4]:
import altair as alt
import pandas as pd

from pyclick.click_models import PBM, CM

from ultr_toolbox.data import ClickDataset
from ultr_toolbox.models.em import EMTrainer
from ultr_toolbox.models.neural import NeuralTrainer, PositionBasedModel, CascadeModel
from ultr_toolbox.models.ctr import CTRTrainer, RandomModel, DocumentBasedModel, RankBasedModel, RankDocumentBasedModel, JointModel

In [5]:
from sklearn.model_selection import train_test_split

path = "data/clicks.parquet"
df = pd.read_parquet(path)
df = df.head(5_000_000)
train_df, test_df = train_test_split(df)
train_df, val_df = train_test_split(train_df)

In [6]:
n_documents = 1_000_000
n_ranks = 10

In [7]:
train_dataset = ClickDataset(train_df)
val_dataset = ClickDataset(val_df)
test_dataset = ClickDataset(test_df)

In [8]:
models = {
    "RandomModel": CTRTrainer(RandomModel()),
    "RankBasedModel": CTRTrainer(RankBasedModel()),
    "DocumentBasedModel": CTRTrainer(DocumentBasedModel()),
    "RankDocumentBasedModel": CTRTrainer(RankDocumentBasedModel()),
    "JointModel": CTRTrainer(JointModel()),
    "PositionBasedModel": NeuralTrainer(PositionBasedModel(n_documents=n_documents, n_ranks=n_ranks)),
    "CascadeModel": NeuralTrainer(CascadeModel(n_documents=n_documents)),
    "EM-PBM": EMTrainer(PBM()),
    "EM-Cascade": EMTrainer(CM()),
}

In [9]:
metrics = []

for model, trainer in models.items():
    trainer.train(train_dataset, val_dataset)
    metric = trainer.test(test_dataset)
    metric["model"] = model
    metrics.append(metric)

test_df = pd.DataFrame(metrics)

Testing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9766/9766 [00:02<00:00, 3600.40it/s]
/Users/philipphager/miniconda3/envs/ultr-toolbox/lib/python3.9/site-packages/numpy/core/_methods.py:48: RuntimeWarning: divide by zero encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
Testing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1250000/1250000 [02:26<00:00, 8558.12it/s]


In [10]:
alt.Chart(test_df, width=800, height=400).mark_bar(clip=True).encode(
    x=alt.X("model", sort="y", title="", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("perplexity", title="Perplexity", scale=alt.Scale(domain=[1, 1.6])),
    color="model",
    tooltip=["perplexity"]
)

alt.Chart(...)